In [28]:
import os
import numpy as np
import pandas as pd
from tsfresh import extract_features
from tsfresh.feature_extraction import ComprehensiveFCParameters
# import roll
# Load the preprocessed datasets
X_test = pd.read_csv('data/processed/X_test_transformed.csv', index_col='Date', parse_dates=True)
y_test = pd.read_csv('data/processed/y_test_transformed.csv', index_col='Date', parse_dates=True)
X_train = pd.read_csv('data/processed/X_train_transformed.csv', index_col='Date', parse_dates=True)
y_train = pd.read_csv('data/processed/y_train_transformed.csv', index_col='Date', parse_dates=True)

# Combine X and y dataframes for feature engineering
train_combined = pd.concat([X_train, y_train], axis=1)
test_combined = pd.concat([X_test, y_test], axis=1)

# Ensure the column names are preserved
base_features = list(X_train.columns)

# Add ID and time columns required by TSFRESH
train_combined['id'] = 1
train_combined['time'] = train_combined.index

test_combined['id'] = 2
test_combined['time'] = test_combined.index


# Load the Fc Params
fc_parameters = ComprehensiveFCParameters()

# Automated Feature Extraction using TSFRESH
from tsfresh.utilities.dataframe_functions import roll_time_series
def extract_tsfresh_features(data, column_id, column_sort, default_fc_parameters):
    df_long = roll_time_series(data, column_id=column_id, column_sort=column_sort)
    extracted_features = extract_features(df_long, column_id=column_id, column_sort=column_sort, default_fc_parameters=default_fc_parameters)
    extracted_features = extracted_features.dropna(axis=1, how='any')  # Drop columns with NaNs
    return extracted_features

# Actually extract/engineer the features
tsfresh_features_train = extract_tsfresh_features(train_combined, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)
tsfresh_features_train.index = train_combined.index[:len(tsfresh_features_train)]

tsfresh_features_test = extract_tsfresh_features(test_combined, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)
tsfresh_features_test.index = test_combined.index[:len(tsfresh_features_test)]


all_added_features = []

tsfresh_features_train.to_csv('data/tsfresh/train_combined_all_features')
tsfresh_features_test.to_csv('data/tsfresh/test_combined_all_features')


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\tsfresh\utilities\dataframe_functions.py:520: UserWarning: Your time stamps are not uniformly sampled, which makes rolling nonsensical in some domains.
  warnings.warn(
Feature Extraction: 100%|██████████| 20/20 [09:53<00:00, 29.67s/it]
c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\tsfresh\utilities\dataframe_functions.py:520: UserWarning: Your time stamps are not uniformly sampled, which makes rolling nonsensical in some domains.
  warnings.warn(
Feature Extraction: 100%|██████████| 20/20 [00:53<00:00,  2.65s/it]


In [40]:
import os
import pandas as pd

# Load the saved TSFRESH features
tsfresh_features_train = pd.read_csv('data/tsfresh/train_combined_all_features.csv', index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv('data/tsfresh/test_combined_all_features.csv', index_col='Date', parse_dates=True)

# Verify the structure of the loaded data
print(f"Training data shape: {tsfresh_features_train.shape}")
print(f"Testing data shape: {tsfresh_features_test.shape}")

# Display the first few rows to ensure correct loading
display(tsfresh_features_train.head())
display(tsfresh_features_test.head())

# Forward fill any missing values
tsfresh_features_train_filled = tsfresh_features_train.ffill()
tsfresh_features_test_filled = tsfresh_features_test.ffill()

# Save the forward-filled DataFrames back to the TSFRESH subdirectory
tsfresh_features_train_filled.to_csv('data/tsfresh/train_combined_all_features_filled.csv')
tsfresh_features_test_filled.to_csv('data/tsfresh/test_combined_all_features_filled.csv')

print("Forward filling complete and data saved.")


Training data shape: (460, 6082)
Testing data shape: (115, 6082)


,FEDFUNDS__variance_larger_than_standard_deviation,FEDFUNDS__has_duplicate_max,FEDFUNDS__has_duplicate_min,FEDFUNDS__has_duplicate,FEDFUNDS__sum_values,FEDFUNDS__abs_energy,FEDFUNDS__median,FEDFUNDS__mean,FEDFUNDS__length,FEDFUNDS__standard_deviation,...,EXPGS__ratio_beyond_r_sigma__r_6,EXPGS__ratio_beyond_r_sigma__r_7,EXPGS__ratio_beyond_r_sigma__r_10,EXPGS__count_above__t_0,EXPGS__count_below__t_0,EXPGS__lempel_ziv_complexity__bins_2,EXPGS__lempel_ziv_complexity__bins_3,EXPGS__lempel_ziv_complexity__bins_5,EXPGS__lempel_ziv_complexity__bins_10,EXPGS__lempel_ziv_complexity__bins_100
Date,,,,,,,,,,,,,,,,,,,,,
1976-02-01,0.0,0.0,0.0,0.0,0.161211,0.025989,0.161211,0.161211,1.0,0.000000,...,0.0,0.0,0.0,0.0,1.0,1.000000,1.00,1.00,1.00,1.0
1976-03-01,0.0,0.0,0.0,0.0,0.064030,0.035433,0.032015,0.032015,2.0,0.129196,...,0.0,0.0,0.0,0.0,1.0,1.000000,1.00,1.00,1.00,1.0
1976-04-01,0.0,0.0,0.0,0.0,0.094785,0.036379,0.030755,0.031595,3.0,0.105490,...,0.0,0.0,0.0,0.0,1.0,0.666667,1.00,1.00,1.00,1.0
1976-05-01,0.0,0.0,0.0,0.0,-0.380434,0.262212,-0.033213,-0.095108,4.0,0.237713,...,0.0,0.0,0.0,0.0,1.0,0.500000,0.75,0.75,0.75,1.0
1976-06-01,0.0,0.0,1.0,1.0,-0.855652,0.488045,-0.097181,-0.171130,5.0,0.261387,...,0.0,0.0,0.0,0.0,1.0,0.400000,0.80,0.80,0.80,1.0


,FEDFUNDS__variance_larger_than_standard_deviation,FEDFUNDS__has_duplicate_max,FEDFUNDS__has_duplicate_min,FEDFUNDS__has_duplicate,FEDFUNDS__sum_values,FEDFUNDS__abs_energy,FEDFUNDS__median,FEDFUNDS__mean,FEDFUNDS__length,FEDFUNDS__standard_deviation,...,EXPGS__ratio_beyond_r_sigma__r_6,EXPGS__ratio_beyond_r_sigma__r_7,EXPGS__ratio_beyond_r_sigma__r_10,EXPGS__count_above__t_0,EXPGS__count_below__t_0,EXPGS__lempel_ziv_complexity__bins_2,EXPGS__lempel_ziv_complexity__bins_3,EXPGS__lempel_ziv_complexity__bins_5,EXPGS__lempel_ziv_complexity__bins_10,EXPGS__lempel_ziv_complexity__bins_100
Date,,,,,,,,,,,,,,,,,,,,,
2014-07-01,0.0,0.0,0.0,0.0,-0.001855,0.000003,-0.001855,-0.001855,1.0,0.000000,...,0.0,0.0,0.0,0.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000
2014-08-01,0.0,1.0,1.0,1.0,-0.003710,0.000007,-0.001855,-0.001855,2.0,0.000000,...,0.0,0.0,0.0,0.0,1.0,0.500000,0.500000,0.500000,0.500000,0.500000
2014-09-01,0.0,0.0,1.0,1.0,-0.003710,0.000007,-0.001855,-0.001237,3.0,0.000875,...,0.0,0.0,0.0,0.0,1.0,0.666667,0.666667,0.666667,0.666667,0.666667
2014-10-01,0.0,1.0,1.0,1.0,-0.003710,0.000007,-0.000928,-0.000928,4.0,0.000928,...,0.0,0.0,0.0,0.0,1.0,0.750000,0.750000,0.750000,0.750000,0.750000
2014-11-01,0.0,1.0,1.0,1.0,-0.003710,0.000007,0.000000,-0.000742,5.0,0.000909,...,0.0,0.0,0.0,0.0,1.0,0.600000,0.800000,0.800000,0.800000,0.800000


Forward filling complete and data saved.


In [35]:
# Verify that the columns are consistent between train and test
train_columns = set(tsfresh_features_train.columns)
test_columns = set(tsfresh_features_test.columns)

# Check if there's any difference between train and test columns
column_difference = train_columns.symmetric_difference(test_columns)
if not column_difference:
    print("Columns in training and testing datasets match.")
else:
    print(f"Columns differ between train and test datasets: {column_difference}")


Columns in training and testing datasets match.


In [39]:
non_nan_count = tsfresh_features_train.isna().sum().sum()
print(f"Total -NaN values in the training data: {non_nan_count}")


Total -NaN values in the training data: 0


In [38]:
non_nan_count = tsfresh_features_train.notna().sum().sum()
print(f"Total number of non-NaN values in the training data: {non_nan_count}")


Total number of non-NaN values in the training data: 2797720


In [29]:
len(tsfresh_features_train.columns)

6082

In [ ]:

X_train_all = extract_features(X_train,column_id="id",column_sort="time",default_fc_parameters=ComprehensiveFCParameters())

In [23]:
X_train = pd.read_csv('data/engineered/X_train_all_features.csv')

display(X_train)

,Unnamed: 0,FEDFUNDS__value__variance_larger_than_standard_deviation,FEDFUNDS__value__has_duplicate_max,FEDFUNDS__value__has_duplicate_min,FEDFUNDS__value__has_duplicate,FEDFUNDS__value__sum_values,FEDFUNDS__value__abs_energy,FEDFUNDS__value__mean_abs_change,FEDFUNDS__value__mean_change,FEDFUNDS__value__mean_second_derivative_central,...,USREC__value__lempel_ziv_complexity__bins_3,USREC__value__lempel_ziv_complexity__bins_5,USREC__value__lempel_ziv_complexity__bins_10,USREC__value__lempel_ziv_complexity__bins_100,USREC__value__permutation_entropy__dimension_3__tau_1,USREC__value__permutation_entropy__dimension_4__tau_1,USREC__value__permutation_entropy__dimension_5__tau_1,USREC__value__permutation_entropy__dimension_6__tau_1,USREC__value__permutation_entropy__dimension_7__tau_1,USREC__value__mean_n_absolute_max__number_of_maxima_7
0,1.0,0.0,1.0,1.0,1.0,-0.954674,21.11444,0.146979,-0.000351,0.000284,...,0.063043,0.063043,0.063043,0.063043,-0.0,-0.0,-0.0,-0.0,-0.0,0.0


In [15]:
# Load all Fc Params once, and save the full dataset as a csv file
import os
import numpy as np
import pandas as pd
from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters

# Load the preprocessed datasets
X_test = pd.read_csv('data/processed/X_test_transformed.csv', index_col='Date', parse_dates=True)
y_test = pd.read_csv('data/processed/y_test_transformed.csv', index_col='Date', parse_dates=True)
X_train = pd.read_csv('data/processed/X_train_transformed.csv', index_col='Date', parse_dates=True)
y_train = pd.read_csv('data/processed/y_train_transformed.csv', index_col='Date', parse_dates=True)

# Load the Fc Params
fc_parameters = EfficientFCParameters()

# Prepare the data for feature extraction
def prepare_data_for_tsfresh(X):
    X['id'] = 1
    X['time'] = X.index
    return X

X_train_prepared = prepare_data_for_tsfresh(X_train)
X_test_prepared = prepare_data_for_tsfresh(X_test)

# Extract the features using ComprehensiveFCParameters
X_train_features = extract_features(X_train_prepared, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)
X_test_features = extract_features(X_test_prepared, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)

# Save the engineered features to CSV files
X_train_features.to_csv('data/engineered/X_train_all_features.csv')
X_test_features.to_csv('data/engineered/X_test_all_features.csv')

print("Engineered features have been saved to CSV files.")



Feature Extraction: 100%|██████████| 14/14 [00:04<00:00,  2.82it/s]


Engineered features have been saved to CSV files.


In [16]:
X_train = pd.read_csv('data/engineered/X_train_all_features.csv')

display(X_train)

,Unnamed: 0,FEDFUNDS__variance_larger_than_standard_deviation,FEDFUNDS__has_duplicate_max,FEDFUNDS__has_duplicate_min,FEDFUNDS__has_duplicate,FEDFUNDS__sum_values,FEDFUNDS__abs_energy,FEDFUNDS__mean_abs_change,FEDFUNDS__mean_change,FEDFUNDS__mean_second_derivative_central,...,CUMFNS__fourier_entropy__bins_5,CUMFNS__fourier_entropy__bins_10,CUMFNS__fourier_entropy__bins_100,CUMFNS__permutation_entropy__dimension_3__tau_1,CUMFNS__permutation_entropy__dimension_4__tau_1,CUMFNS__permutation_entropy__dimension_5__tau_1,CUMFNS__permutation_entropy__dimension_6__tau_1,CUMFNS__permutation_entropy__dimension_7__tau_1,CUMFNS__query_similarity_count__query_None__threshold_0.0,CUMFNS__mean_n_absolute_max__number_of_maxima_7
0,1,0.0,1.0,1.0,1.0,-0.954674,21.11444,0.146979,-0.000351,0.000284,...,1.02575,1.665036,3.608037,1.780438,3.123936,4.568559,5.660642,6.030141,NaN,0.797222


In [17]:
# Load all Fc Params once, and save the full dataset as a csv file
import os
import numpy as np
import pandas as pd
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters

# Load the preprocessed datasets
X_test = pd.read_csv('data/processed/X_test_transformed.csv', index_col='Date', parse_dates=True)
y_test = pd.read_csv('data/processed/y_test_transformed.csv', index_col='Date', parse_dates=True)
X_train = pd.read_csv('data/processed/X_train_transformed.csv', index_col='Date', parse_dates=True)
y_train = pd.read_csv('data/processed/y_train_transformed.csv', index_col='Date', parse_dates=True)

# Load the Fc Params
fc_parameters = MinimalFCParameters()

# Prepare the data for feature extraction
def prepare_data_for_tsfresh(X):
    X['id'] = 1
    X['time'] = X.index
    return X

X_train_prepared = prepare_data_for_tsfresh(X_train)
X_test_prepared = prepare_data_for_tsfresh(X_test)

# Extract the features using ComprehensiveFCParameters
X_train_features = extract_features(X_train_prepared, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)
X_test_features = extract_features(X_test_prepared, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)

# Save the engineered features to CSV files
X_train_features.to_csv('data/engineered/X_train_all_features.csv')
X_test_features.to_csv('data/engineered/X_test_all_features.csv')

print("Engineered features have been saved to CSV files.")



Feature Extraction: 100%|██████████| 14/14 [00:04<00:00,  3.33it/s]

Engineered features have been saved to CSV files.


In [18]:
X_train = pd.read_csv('data/engineered/X_train_all_features.csv')

display(X_train)

,Unnamed: 0,FEDFUNDS__sum_values,FEDFUNDS__median,FEDFUNDS__mean,FEDFUNDS__length,FEDFUNDS__standard_deviation,FEDFUNDS__variance,FEDFUNDS__root_mean_square,FEDFUNDS__maximum,FEDFUNDS__absolute_maximum,...,CUMFNS__sum_values,CUMFNS__median,CUMFNS__mean,CUMFNS__length,CUMFNS__standard_deviation,CUMFNS__variance,CUMFNS__root_mean_square,CUMFNS__maximum,CUMFNS__absolute_maximum,CUMFNS__minimum
0,1,-0.954674,0.0,-0.002075,460.0,0.214235,0.045897,0.214245,0.472131,0.475219,...,-8.088906,-0.007069,-0.017585,460.0,0.345797,0.119576,0.346244,0.773459,0.797222,-0.797222
